# 로컬 LLM으로 법안 적용 기준 뽑기 (12주차: 생성형 AI 이해 및 로컬 LLM 구성)

검토 에이전트(`server/api/agent.js`)는 법안 제안이유에서 건물 적용 기준(용도·연면적·층수·사용승인·위임)을 **OpenAI gpt-4o-mini**로 뽑고, 근거 문구가 원문에 그대로 있는지 코드로 대조한다(`server/lib/criteria.js`).
이 노트북은 같은 일을 **내 컴퓨터(또는 Colab)에서 돌리는 공개 모델**로 해 보고, 시범 법안 26건에서 OpenAI 결과와 나란히 비교한다.

**순서**: 0 왜 로컬 LLM인가 → 1 문제 → 2 데이터 → 3 모델 불러오기 → 4 프롬프트 → 5 추출 실행 → 6 원문 대조 → 7 정답 비교 → 8 결과표 → 9 한계 → 10 화면·서버와의 관계

## 0. 왜 로컬 LLM인가
- **행정망 망분리**: 지자체 업무망은 인터넷과 분리돼 있어 외부 AI API를 부를 수 없는 경우가 많다. 모델을 청사 안 서버에 두면 망 안에서 쓸 수 있다.
- **법안 원문 외부 전송 없음**: 법안은 공개 자료지만, 담당자 메모·검토 초안·내부 자료가 함께 나가지 않게 하려면 모델이 기관 안에 있는 편이 안전하다.
- **비용**: 호출마다 돈이 드는 API 대신, 한 번 받은 모델을 CPU에서 계속 쓸 수 있다 (대신 느리고 정확도가 낮을 수 있다, 9절).

In [ ]:
# 준비: Colab 에는 torch·transformers 가 이미 있다. GitHub Actions(CPU)에는 없으므로 CPU 전용 torch 를 받는다 (GPU 판보다 훨씬 작음)
import os, sys, importlib.util
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'          # 사용 통계 보내지 않음
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if importlib.util.find_spec('torch') is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
if importlib.util.find_spec('transformers') is None:
    !pip install -q "transformers>=4.45,<5"
if 'google.colab' in sys.modules and not os.path.exists('/usr/share/fonts/truetype/nanum'):
    !apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import os, re, json, math, time, datetime, urllib.request
import pandas as pd
from IPython.display import Markdown, display
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
START = time.time()

## 1. 문제
- **입력**: 법안 제안이유 전문 (한국어, 법안마다 300~900자)
- **출력**: 건물 기준 목록 `{"items":[{"attr","bound","value","quote"}]}` (서버 에이전트와 같은 형식)
- **채점**: 모델이 낸 기준(후보)마다 서버와 **같은 규칙**으로 원문 대조한다. 근거 문구(quote)가 원문에 한 글자도 다르지 않게 있어야 하고, 수치·용도 이름·위임 문구가 그 문구 안에 있어야 통과다. 통과한 것만 규칙이 된다.
- **정답**: 사람이 원문과 대조해 만든 판정 시범 규칙 26건 (`web/data/demo_rules.json`). 지도 화면의 시범 판정에 쓰는 규칙이다.
- **비교 대상**: 같은 지시문으로 돌린 OpenAI gpt-4o-mini (배포된 에이전트에서 수집, `data/llm_compare_openai.csv`).

## 2. 데이터
GitHub 저장소(main)에서 읽고, 안 되면 저장소 안의 파일(`../`)을 읽는다.

| 파일 | 내용 |
|---|---|
| `web/data/demo_rules.json` | 판정 시범 규칙 26건 (정답) |
| `web/data/summaries.json` | 법안 제안이유 전문 (의안번호 → 글) |
| `server/lib/use-terms.js`, `server/lib/judge.js` | 주용도 코드별 시설 이름표, 코드 이름 (서버 원문 대조가 쓰는 것 그대로) |
| `data/verify_cases.json` | 서버 `verify()` 결과를 담은 시험 사례 240개 + 서버의 기준 뽑기 지시문 (`server/scripts/verify-cases.js` 가 만듦) |
| `data/llm_compare_openai.csv` | 배포된 에이전트(gpt-4o-mini)에서 모은 같은 26건의 추출 결과 |

In [ ]:
RAW = 'https://raw.githubusercontent.com/cbnu-lawmap/law-impact-map/main/'

def read_text(rel):
    """저장소 파일 하나를 글로 읽는다: GitHub main → 저장소 폴더(../)"""
    try:
        with urllib.request.urlopen(RAW + rel, timeout=30) as r:
            return r.read().decode('utf-8-sig')
    except Exception:
        roots = [os.path.dirname(os.environ.get('NB_LOCAL_DATA', '').rstrip('/')), '..', '.']
        for root in roots:
            p = os.path.join(root, rel)
            if root and os.path.exists(p):
                return open(p, encoding='utf-8-sig').read()
        raise FileNotFoundError(rel + ' 을 찾지 못했습니다 (저장소를 받아 notebooks/ 에서 실행하거나 GitHub 에 올라간 뒤 실행)')

demos = json.loads(read_text('web/data/demo_rules.json'))['demos']
full = json.loads(read_text('web/data/summaries.json'))['summaries']
src = read_text('server/lib/use-terms.js')
USE_TERMS = json.loads(src[src.index('{'):src.rindex('}') + 1])
judge_js = read_text('server/lib/judge.js')
USE_NAMES = dict(re.findall(r'"(\d{5})":\s*"([^"]+)"', re.search(r'USE_NAMES = (\{.*?\});', judge_js, re.S).group(1)))
VC = json.loads(read_text('data/verify_cases.json'))

bills = pd.DataFrame([{'bill_no': d['bill_no'], 'title': d['title'], 'text': full.get(d['bill_no'], ''), 'gold': d['rule']} for d in demos])
bills['chars'] = bills.text.str.len()
print('시범 법안', len(bills), '건 / 제안이유 전문 있음', (bills.chars > 0).sum(), '건 / 글자 수 평균 %.0f, 최대 %d' % (bills.chars.mean(), bills.chars.max()))
print('주용도 코드', len(USE_NAMES), '개, 시설 이름표', sum(len(v) for v in USE_TERMS.values()), '개')
bills[['bill_no', 'title', 'chars', 'gold']].head(8)

## 3. 모델 불러오기
**Qwen/Qwen2.5-1.5B-Instruct** (Alibaba Qwen, 15억 매개변수, **Apache License 2.0**, 상업 사용 가능, 토큰 없이 받을 수 있음)
- 한국어를 포함한 29개 이상 언어로 지시를 따르도록 학습된 공개 모델이고, JSON 같은 구조화 출력을 잘하는 편이라 골랐다.
- 크기: 약 3.1GB(bf16 가중치). CPU에서는 float32로 올려 약 6GB 메모리를 쓴다. GPU(Colab T4)가 있으면 float16으로 올린다.
- 더 작은 0.5B는 빠르지만 원문을 그대로 옮기는 능력이 크게 떨어지고, 3B 이상은 GitHub Actions CPU에서 26건을 시간 안에 끝내기 어렵다. 다른 모델을 쓰려면 `LOCAL_LLM_MODEL` 환경변수로 바꾼다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers

MODEL_ID = os.environ.get('LOCAL_LLM_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
LICENSE = 'Apache-2.0' if MODEL_ID.startswith('Qwen/Qwen2.5-') and '-3B' not in MODEL_ID and '-72B' not in MODEL_ID else '모델 카드 확인'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
if DEVICE == 'cpu':
    torch.set_num_threads(os.cpu_count() or 1)
torch.manual_seed(0)

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'                 # 여러 법안을 한 번에 생성할 때 왼쪽을 채운다
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
LOAD_SEC = time.time() - t0
n_params = sum(p.numel() for p in model.parameters())
RUNTIME = f'{DEVICE.upper()} {torch.get_num_threads()}스레드' if DEVICE == 'cpu' else torch.cuda.get_device_name(0)
print(f'{MODEL_ID} ({LICENSE}) · 매개변수 {n_params/1e9:.2f}B · {DTYPE} · {RUNTIME} · 불러오기 {LOAD_SEC:.0f}초')
print('torch', torch.__version__, '/ transformers', transformers.__version__)

## 4. 프롬프트
서버 에이전트의 `extract_criteria` 가 쓰는 지시문을 **그대로** 쓴다. 지시문은 서버 코드(`EXTRACT_PROMPT`)에서 `data/verify_cases.json` 으로 내보낸 것을 읽으므로 글자 하나까지 같다.
사용자 메시지도 서버와 같이 `[요약]\n` + 제안이유 전문이고, 출력 상한도 서버와 같은 500토큰이다.

다른 점 (공정성을 위해 적어 둠)
- OpenAI는 `response_format: json_object` 로 JSON만 내도록 강제되지만, 로컬 모델에는 그 기능이 없다. 그래서 출력에 붙은 코드 블록 표시(```)를 떼고 첫 `{` 부터 마지막 `}` 까지를 JSON으로 읽는다. 그래도 읽지 못하면 후보 0개로 센다 (서버는 이때 정규식 추출로 대신하지만, 여기서는 모델 실력만 본다).
- 서버는 temperature 0.2, 여기서는 다시 돌려도 같은 결과가 나오게 탐욕적 생성(temperature 0)을 쓴다.

In [ ]:
SYSTEM = VC['extract_prompt']
USER_PREFIX = VC['extract_user_prefix']
MAX_NEW_TOKENS = VC['extract_max_tokens']          # 서버 CALL_MAX_TOKENS 와 같음
REF_DATE = VC['ref_date']
assert SYSTEM.startswith('법안 요약에서 건축물 적용 기준만 JSON') and REF_DATE == '2026-11-01'

def chat_prompt(text):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': USER_PREFIX + text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(SYSTEM)
print()
n_in = [len(tok(chat_prompt(t)).input_ids) for t in bills.text]
print('입력 토큰: 평균 %.0f, 최대 %d (지시문만 %d)' % (sum(n_in) / len(n_in), max(n_in), len(tok(SYSTEM).input_ids)))

## 5. 추출 실행
- 26건을 글 길이 순으로 묶어(묶음 크기 CPU 9, GPU 13) 한 번에 생성한다. CPU에서는 한 토큰을 만들 때마다 가중치 6GB를 읽어야 해서, 한 건씩보다 묶어서 만드는 편이 훨씬 빠르다.
- 시간 상한 `TIME_BUDGET` (기본 45분)을 넘을 것 같으면 남은 묶음은 돌리지 않고 "시간 초과"로 적는다 (GitHub Actions 실행 시간 제한).

In [ ]:
TIME_BUDGET = float(os.environ.get('LOCAL_LLM_BUDGET_MIN', '45')) * 60
BATCH = 9 if DEVICE == 'cpu' else 13

def parse_items(s):
    """모델 출력 → items 목록. 못 읽으면 None"""
    s = re.sub(r'^\s*```(?:json)?\s*|\s*```\s*$', '', s.strip())
    for cand in [s, s[s.find('{'): s.rfind('}') + 1] if '{' in s else '']:
        try:
            j = json.loads(cand)
        except Exception:
            continue
        items = j.get('items') if isinstance(j, dict) else None
        if isinstance(items, list):
            return items
    return None

order = bills.chars.sort_values().index.tolist()
raw = {}
gen_sec = {}
t_run = time.time()
for b in range(0, len(order), BATCH):
    idx = order[b:b + BATCH]
    done = time.time() - t_run
    if b and done / b * (b + len(idx)) > TIME_BUDGET:
        print(f'시간 상한 {TIME_BUDGET/60:.0f}분을 넘을 것 같아 {len(order) - b}건은 돌리지 않음')
        break
    prompts = [chat_prompt(bills.text[i]) for i in idx]
    enc = tok(prompts, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    sec = time.time() - t0
    for k, i in enumerate(idx):
        raw[i] = tok.decode(out[k, enc.input_ids.shape[1]:], skip_special_tokens=True)
        gen_sec[i] = sec / len(idx)
    print(f'묶음 {b // BATCH + 1}: {len(idx)}건 {sec:.0f}초 (입력 최대 {enc.input_ids.shape[1]}토큰, 생성 {out.shape[1] - enc.input_ids.shape[1]}토큰)')

RUN_SEC = time.time() - t_run
bills['raw'] = [raw.get(i) for i in bills.index]
bills['items'] = [parse_items(r) if isinstance(r, str) else None for r in bills.raw]
bills['status'] = ['시간 초과' if not isinstance(r, str) else ('JSON 실패' if it is None else '완료') for r, it in zip(bills.raw, bills['items'])]
print(f'추출 {RUN_SEC/60:.1f}분, 법안당 평균 {RUN_SEC / max(1, len(raw)):.0f}초')
print(bills.status.value_counts().to_string())

In [ ]:
# 출력 예: 첫 법안
i0 = bills.index[0]
print(bills.bill_no[i0], bills.title[i0])
print(bills.raw[i0] if isinstance(bills.raw[i0], str) else '(돌리지 않음)')

## 6. 원문 대조
`server/lib/criteria.js` 의 `verify()` 를 파이썬으로 그대로 옮겼다. 규칙은 서버와 같다.
1. 근거 문구(quote)가 제안이유 원문(공백만 맞춤)에 **그대로** 있어야 한다.
2. 연면적·층수 수치는 근거 문구에 나와야 한다 (1000 → 1000, 1,000, 1천 / 100000 → 10만). 연면적은 면적 단위, 층수는 "N층" 이 있어야 한다.
3. 사용승인은 기준일(2026-11-01) − N년이고 근거에 "N년" 이 있어야 한다.
4. 용도는 코드마다 그 시설 이름(별표1 이름·하위 시설 예, `data/building_use_codes.csv`)이 근거에 있어야 한다.
5. 위임은 근거에 "대통령령·부령·조례로 정하는" 같은 위임 문구가 있고, 값이 그 문구 안의 짧은 기준 이름(15자 이하)이어야 한다.

옮긴 코드가 서버와 같은지 먼저 확인한다: 서버 `verify()` 가 낸 결과를 담은 시험 사례(`data/verify_cases.json`)를 파이썬으로 다시 돌려 모두 같아야 한다.

In [ ]:
REF_DATE = '2026-11-01'           # server/lib/criteria.js 의 REF_DATE
NOT_IN_QUOTE = '근거 문장에 없는 값 제외'
DELEG_NO_PHRASE = '근거 문장에 위임 문구(대통령령·부령·조례로 정하는) 없음'
DELEG_LONG_VALUE = '위임 값이 기준 이름이 아님 (너무 긺)'
DELEG_VALUE_MAX = 15
AREA_UNIT = re.compile(r'제곱미터|㎡|평방미터|m2|m²')
DELEG_RE = re.compile(r'(대통령령|총리령|[가-힣]{1,10}부령|[가-힣]{0,6}조례)(?:으로|로)\s*정(?:하는|한다|함|할|해)([^.]{0,30}?(용도|규모|기준|범위|요건))?')
_WS = r'[\s﻿ ]'

def norm_text(s):
    s = '' if s is None or s == '' else js_str(s)
    return re.sub(_WS + '+', ' ', s).strip(' ')

def js_num_str(x):
    """자바스크립트 String(숫자) 와 같게 (1000.0 → '1000')"""
    if isinstance(x, bool): return 'true' if x else 'false'
    if isinstance(x, float):
        if math.isnan(x): return 'NaN'
        if math.isinf(x): return 'Infinity' if x > 0 else '-Infinity'
        if x.is_integer(): return str(int(x))
        return repr(x)
    return str(x)

def js_str(x):
    if x is None: return 'null'
    if isinstance(x, (bool, int, float)): return js_num_str(x)
    if isinstance(x, list): return ','.join('' if v is None else js_str(v) for v in x)
    if isinstance(x, dict): return '[object Object]'
    return str(x)

def js_number(x):
    """자바스크립트 Number(String(x).replace(/,/g, '')) 와 같게"""
    s = js_str(x).replace(',', '').strip()
    if s == '': return 0
    if s in ('Infinity', '+Infinity', '-Infinity'): return float(s.replace('Infinity', 'inf'))
    if not re.fullmatch(r'[+-]?([0-9]+\.?[0-9]*|\.[0-9]+)([eE][+-]?[0-9]+)?', s): return float('nan')
    v = float(s)
    return int(v) if v.is_integer() and abs(v) < 2**53 else v

def number_forms(n):
    forms = [js_num_str(n)]
    if float(n).is_integer():
        forms.append(f'{int(n):,}')
    else:
        forms.append(f'{n:,.3f}'.rstrip('0').rstrip('.'))
    if float(n).is_integer():
        n = int(n)
        if n >= 10000 and n % 10000 == 0: forms.append(js_num_str(n / 10000) + '만')
        if 1000 <= n < 10000 and n % 1000 == 0: forms.append(js_num_str(n / 1000) + '천')
    return list(dict.fromkeys(forms))

def number_in_text(n, text):
    return any(re.search(r'(?<![0-9,])' + re.escape(f) + r'(?![0-9,])', text) for f in number_forms(n))

def use_term_in(code, quote, use_terms):
    q = re.sub(_WS + '+', '', js_str(quote))
    for t in use_terms.get(js_str(code), []):
        if t in q: return t
    return None

def verify(candidates, summary, use_names, use_terms):
    """server/lib/criteria.js 의 verify() 를 그대로 옮김: 근거 문구가 원문에 그대로 있고, 값이 그 문구에 있어야 통과"""
    text = norm_text(summary)
    rule, evidence, dropped = {}, [], []
    for c0 in candidates:
        c = dict(c0) if isinstance(c0, dict) else {}
        attr = c.get('attr')
        if attr in ('연면적', '층수'): c['value'] = js_number(c.get('value'))
        if attr == '용도' and not isinstance(c.get('value'), list): c['value'] = [js_str(c.get('value'))]
        if attr == '위임': c['value'] = js_str(c.get('value'))
        value = c.get('value')
        quote = norm_text(c.get('quote'))
        bound = c.get('bound')
        why = ''
        num_bad = attr in ('연면적', '층수') and (not math.isfinite(value) or value <= 0 or bound not in ('min', 'max'))
        if attr not in ('용도', '연면적', '층수', '사용승인', '위임'): why = '모르는 기준'
        elif num_bad: why = '수치나 경계(min·max)가 올바르지 않음'
        elif not quote or quote not in text: why = '근거 문구가 요약 원문에 그대로 없음'
        elif attr in ('연면적', '층수') and not number_in_text(value, quote): why = '수치 ' + js_num_str(value) + ' 가 근거 문구에 없음'
        elif attr == '사용승인':
            years = int(REF_DATE[:4]) - js_number(js_str(value)[:4])
            ys = js_num_str(years)
            if js_str(value)[4:] != REF_DATE[4:] or not re.search(r'(?<![0-9])' + ys + r'\s*년', quote):
                why = "근거 문구에 '" + ys + "년' 이 없음"
        elif attr == '용도' and not (isinstance(value, list) and all(js_str(v) in use_names for v in value)): why = '모르는 용도 코드'
        elif attr == '연면적' and not AREA_UNIT.search(quote): why = NOT_IN_QUOTE + ' (면적 단위)'
        elif attr == '층수' and not re.search(r'(?<![0-9,])' + js_num_str(value) + r'\s*층', quote): why = NOT_IN_QUOTE + ' (' + js_num_str(value) + '층)'
        elif attr == '위임' and re.sub(_WS + '+', '', value) not in re.sub(_WS + '+', '', quote): why = NOT_IN_QUOTE + ' (' + value + ')'
        elif attr == '위임' and not DELEG_RE.search(quote): why = DELEG_NO_PHRASE
        elif attr == '위임' and len(re.sub(_WS + '+', '', value)) > DELEG_VALUE_MAX: why = DELEG_LONG_VALUE
        if why:
            d = {'value': value, 'why': why}
            if attr is not None: d = {'attr': attr, **d}
            dropped.append(d); continue
        if attr == '용도':
            kept = [v for v in value if use_term_in(v, quote, use_terms)]
            gone = [v for v in value if not use_term_in(v, quote, use_terms)]
            if gone:
                dropped.append({'attr': '용도', 'value': gone, 'why': NOT_IN_QUOTE + ' (' + '·'.join(use_names.get(js_str(v), js_str(v)) for v in gone) + ')'})
            if not kept: continue
            value = kept
        if attr in ('용도', '위임'):
            cur = rule.get(attr, [])
            for v in (value if attr == '용도' else [value]):
                if not any(type(v) is type(u) and v == u for u in cur): cur.append(v)
            rule[attr] = cur
        elif attr == '사용승인':
            rule['사용승인'] = {'before': value}
        else:
            rule.setdefault(attr, {})
            if bound in rule[attr]:
                dropped.append({'attr': attr, 'value': value, 'why': '같은 기준이 여러 번 나와 처음 것만 씀'}); continue
            rule[attr][bound] = value
        evidence.append({'attr': attr, 'value': value, 'quote': quote})
    return {'rule': rule, 'evidence': evidence, 'dropped': dropped}


def _plain(x):
    """비교용: NaN → None (자바스크립트 JSON 은 NaN 을 null 로 적음)"""
    if isinstance(x, float) and math.isnan(x): return None
    if isinstance(x, list): return [_plain(v) for v in x]
    if isinstance(x, dict): return {k: _plain(v) for k, v in x.items()}
    return x

bad = [c['name'] for c in VC['cases'] if _plain(verify(c['candidates'], c['text'], USE_NAMES, USE_TERMS)) != c['expected']]
n_cand = sum(len(c['candidates']) for c in VC['cases'])
print(f'서버 verify() 와 비교: 사례 {len(VC["cases"])}개(후보 {n_cand}개) 중 다른 결과 {len(bad)}개')
assert not bad, bad

In [ ]:
def run_verify(row):
    if not isinstance(row['items'], list):
        return {'rule': {}, 'evidence': [], 'dropped': []}
    return verify(row['items'], row['text'], USE_NAMES, USE_TERMS)

bills['v'] = bills.apply(run_verify, axis=1)
bills['cand_n'] = [len(it) if isinstance(it, list) else 0 for it in bills['items']]
bills['pass_n'] = [len(v['evidence']) for v in bills.v]
bills['rule'] = [v['rule'] for v in bills.v]
why = pd.Series([d['why'].split(' (')[0] for v in bills.v for d in v['dropped']], dtype=str)
print('후보', bills.cand_n.sum(), '개 중 원문 대조 통과', bills.pass_n.sum(), '개')
print()
print('버린 이유 (로컬 모델)')
print(why.value_counts().to_string() if len(why) else '없음')

## 7. 정답 비교
통과한 기준으로 만든 규칙을 판정 시범 규칙(정답)과 비교한다. 기준 항목은 용도 코드 하나, 연면적·층수의 경계 하나(예: 연면적 min 100000), 사용승인, 위임 값 하나씩을 한 항목으로 센다.
- **일치**: 규칙이 정답과 똑같음 (모든 항목이 같고 남는 것도 없음)
- **부분 일치**: 정답과 같은 항목이 하나 이상 있지만 빠지거나 남는 항목이 있음
- **틀림**: 통과한 기준은 있지만 정답과 같은 항목이 하나도 없음
- **못 뽑음**: 통과한 기준이 하나도 없음

In [ ]:
def rule_items(rule):
    out = set()
    rule = rule or {}
    for c in rule.get('용도', []) or []: out.add(('용도', str(c)))
    for k in ('연면적', '층수'):
        for b, v in (rule.get(k) or {}).items(): out.add((k, b, float(v)))
    if rule.get('사용승인'): out.add(('사용승인', str(rule['사용승인'].get('before'))))
    for v in rule.get('위임', []) or []: out.add(('위임', re.sub(r'\s+', '', str(v))))
    return out

def grade(rule, gold):
    r, g = rule_items(rule), rule_items(gold)
    if not r: return '못 뽑음'
    if r == g: return '일치'
    return '부분 일치' if r & g else '틀림'

def item_recall(rule, gold):
    g = rule_items(gold)
    return len(rule_items(rule) & g), len(g)

bills['grade'] = [grade(r, g) for r, g in zip(bills.rule, bills.gold)]
GRADES = ['일치', '부분 일치', '틀림', '못 뽑음']
print(bills.grade.value_counts().reindex(GRADES, fill_value=0).to_string())

In [ ]:
# OpenAI (배포된 에이전트 gpt-4o-mini) 결과: 같은 26건, 같은 지시문, 같은 원문 대조
def read_csv(rel):
    try:
        return pd.read_csv(RAW + rel, dtype={'bill_no': str}, encoding='utf-8-sig')
    except Exception:
        return pd.read_csv(os.path.join('..', rel) if os.path.exists(os.path.join('..', rel)) else rel, dtype={'bill_no': str}, encoding='utf-8-sig')

oa = read_csv('data/llm_compare_openai.csv')
oa['rule'] = oa['rule'].apply(json.loads)
oa = bills[['bill_no', 'gold']].merge(oa, on='bill_no', how='left')
oa['grade'] = [grade(r, g) if isinstance(r, dict) else '자료 없음' for r, g in zip(oa.rule, oa.gold)]
OA_MODEL = oa.model.dropna().iloc[0]
OA_DATE = oa.collected_at.dropna().iloc[0]
print('OpenAI 결과:', OA_MODEL, '· 수집', OA_DATE, '·', oa.rule.notna().sum(), '건')

## 8. 결과표
아래 표와 수치는 모두 이 노트북이 실행될 때 코드로 만든다.

In [ ]:
def rule_text(rule):
    if not rule: return '없음'
    parts = []
    if rule.get('용도'): parts.append('용도 ' + '·'.join(USE_NAMES.get(str(c), str(c)) for c in rule['용도']))
    for k, unit in (('연면적', '㎡'), ('층수', '층')):
        for b, v in (rule.get(k) or {}).items():
            parts.append(f'{k} {float(v):,.0f}{unit} ' + ('이상' if b == 'min' else '이하'))
    if rule.get('사용승인'): parts.append('사용승인 ' + str(rule['사용승인'].get('before')) + ' 이전')
    if rule.get('위임'): parts.append('위임 ' + '·'.join(map(str, rule['위임'])))
    return ', '.join(parts)

def summary(cand, passed, rules, grades, golds):
    n = len(grades)
    hit, tot = 0, 0
    for r, g in zip(rules, golds):
        a, b = item_recall(r if isinstance(r, dict) else {}, g); hit += a; tot += b
    gc = pd.Series(grades).value_counts()
    return {
        '모델이 낸 기준 후보': int(cand),
        '원문 대조 통과 후보': int(passed),
        '원문 대조 통과율': f'{passed / cand:.1%}' if cand else '-',
        '기준이 하나라도 통과한 법안': f'{sum(1 for r in rules if isinstance(r, dict) and r)} / {n}',
        '정답 일치': f'{gc.get("일치", 0)} / {n}',
        '부분 일치': f'{gc.get("부분 일치", 0)} / {n}',
        '틀림': f'{gc.get("틀림", 0)} / {n}',
        '못 뽑음': f'{gc.get("못 뽑음", 0)} / {n}',
        '정답 항목 찾은 비율': f'{hit / tot:.1%} ({hit}/{tot})',
    }

done = bills.status != '시간 초과'
local_col = f'로컬 {MODEL_ID.split("/")[-1]}'
oa_col = f'OpenAI {OA_MODEL}'
table = pd.DataFrame({
    local_col: summary(bills.cand_n.sum(), bills.pass_n.sum(), bills.rule.tolist(), bills.grade.tolist(), bills.gold.tolist()),
    oa_col: summary(oa.candidates_n.sum(), oa.passed_n.sum(), oa.rule.tolist(), oa.grade.tolist(), oa.gold.tolist()),
})
table.loc['JSON 형식 실패'] = [f'{(bills.status == "JSON 실패").sum()} / {len(bills)}', '알 수 없음 (서버가 정규식으로 대신함)']
table.loc['돌리지 못함 (시간 초과)'] = [f'{(~done).sum()} / {len(bills)}', '0 / %d' % len(oa)]
table.loc['실행 환경'] = [f'{RUNTIME}, 법안당 {RUN_SEC / max(1, done.sum()):.0f}초 (묶음 생성)', f'OpenAI API (배포된 에이전트, {OA_DATE} 수집)']
table.loc['비용·외부 전송'] = ['없음 (모델 1회 내려받기)', 'API 사용료, 법안 원문이 외부로 전송됨']
display(table)

In [ ]:
# 법안별 비교
detail = pd.DataFrame({
    '의안번호': bills.bill_no,
    '정답(시범 규칙)': [rule_text(g) for g in bills.gold],
    '로컬 통과/후보': [f'{p}/{c}' if s != '시간 초과' else '-' for p, c, s in zip(bills.pass_n, bills.cand_n, bills.status)],
    '로컬 규칙': [rule_text(r) if s == '완료' else s for r, s in zip(bills.rule, bills.status)],
    '로컬': bills.grade,
    'OpenAI 통과/후보': [f'{p}/{c}' for p, c in zip(oa.passed_n.fillna(0).astype(int), oa.candidates_n.fillna(0).astype(int))],
    'OpenAI 규칙': [rule_text(r) if isinstance(r, dict) else '-' for r in oa.rule],
    'OpenAI': oa.grade,
})
display(detail)

In [ ]:
# 결과 저장 (GitHub Actions 가 *_pred.csv 를 같은 브랜치에 커밋한다)
out = pd.DataFrame({
    'bill_no': bills.bill_no,
    'model': MODEL_ID,
    'status': bills.status,
    'candidates_n': bills.cand_n,
    'passed_n': bills.pass_n,
    'rule': [json.dumps(r, ensure_ascii=False) for r in bills.rule],
    'grade': bills.grade,
    'dropped_why': ['; '.join(d['attr'] + ': ' + d['why'] for d in v['dropped'] if d.get('attr')) for v in bills.v],
    'raw': [r[:2000] if isinstance(r, str) else '' for r in bills.raw],
})
data_dir = '../data' if os.path.isdir('../data') else '.'
out.to_csv(os.path.join(data_dir, 'llm_extraction_pred.csv'), index=False, encoding='utf-8-sig')

g_local = bills.grade.value_counts()
g_oa = oa.grade.value_counts()
display(Markdown(
    f"**요약**: {MODEL_ID}({LICENSE})를 {RUNTIME}에서 돌려 26건 중 {done.sum()}건을 {RUN_SEC/60:.0f}분에 처리했다. "
    f"원문 대조 통과율은 로컬 {table.loc['원문 대조 통과율', local_col]}, OpenAI {table.loc['원문 대조 통과율', oa_col]}이고, "
    f"정답과 똑같은 규칙은 로컬 {g_local.get('일치', 0)}건, OpenAI {g_oa.get('일치', 0)}건, "
    f"정답과 겹치는 기준이 있는 법안(일치+부분 일치)은 로컬 {g_local.get('일치', 0) + g_local.get('부분 일치', 0)}건, OpenAI {g_oa.get('일치', 0) + g_oa.get('부분 일치', 0)}건이다. "
    f"(전체 실행 {(time.time() - START)/60:.0f}분)"))

## 9. 한계
- **정답이 적다**: 시범 규칙 26건뿐이고, 사람이 판정용으로 고른 기준이라 위임 값 이름(예: "규모" 와 "연면적")처럼 표현이 갈리는 항목은 맞아도 다르게 셀 수 있다.
- **OpenAI 쪽은 한 번 수집한 값**이다. 배포된 에이전트 응답에는 통과한 근거(evidence)와 버린 개수만 있어서, 후보 수는 "통과 + 버림"으로 셌다 (용도 코드 일부만 버린 후보는 두 번 셀 수 있음). 모델이 JSON을 못 내 서버가 정규식으로 대신한 경우도 구분되지 않는다. 이 PR부터 응답에 `extraction`(방법·후보 수·통과 수·버린 이유)이 붙으므로 다음 수집부터는 정확히 셀 수 있다.
- **같은 조건이 아니다**: OpenAI는 JSON 모드와 temperature 0.2, 로컬은 JSON 모드 없이 탐욕적 생성. 로컬 모델 출력에서 JSON 부분만 떼어 읽는 너그러운 처리를 했다.
- **CPU는 느리다**: 1.5B 모델도 GitHub Actions CPU에서는 법안당 수십 초가 걸린다. 실제 행정망에서는 GPU 서버나 양자화(4bit, llama.cpp·Ollama)로 속도를 높여야 한다.
- **작은 모델의 실수**: 원문을 그대로 옮기지 않고 조금 고쳐 쓰거나(근거 불일치), 주용도 코드를 지어내는 경우가 많다. 원문 대조가 이를 걸러 내므로 **틀린 기준이 판정에 들어가지는 않지만**, 걸러진 만큼 기준을 못 뽑는다.
- 결과는 AI 추정이며 법률 판단이 아니다. 판정에 쓰기 전에 담당자 확인이 필요하다.

## 10. 화면·서버와의 관계
- **화면은 바뀌지 않는다.** 지도·행정 화면은 이 노트북 결과를 읽지 않는다. 결과는 비교 기록(`data/llm_extraction_pred.csv`)으로만 남긴다.
- **서버는 환경변수만으로 로컬 모델을 쓸 수 있다.** `server/api/agent.js` 는 기본이 OpenAI이고, `AGENT_LLM_BASE_URL`(OpenAI 호환 주소, 예: Ollama `http://서버:11434/v1`), `AGENT_LLM_MODEL`(예: `qwen2.5:1.5b-instruct`), `AGENT_LLM_KEY`(선택)를 주면 그 서버로 보낸다. 이때 OpenAI 키는 보내지 않는다.
- **원문 대조는 모델과 상관없이 같다.** 어떤 모델이 뽑든 `verify()` 를 통과한 기준만 판정에 쓰이므로, 작은 로컬 모델로 바꿔도 근거 없는 기준이 지도에 나오지 않는다 (대신 판정 보류·시범 규칙 대체가 늘 수 있다).
- 에이전트 응답의 `extraction` 필드(방법·후보 수·통과 수·버린 이유·규칙)는 이 노트북의 셈과 같아서, 로컬 서버로 돌린 결과도 같은 표로 비교할 수 있다.